In [ ]:
# Cell 1 of 5 -- install and pre-flight check.
#
# pysqldp needs Python >= 3.11, which means Fabric Runtime 1.3 (Spark 3.5) or
# later. Checking it here makes an old runtime fail with a clear message
# instead of an obscure syntax or typing error further down.
#
# If the package is not on a feed this workspace can reach, upload the wheel
# to the Lakehouse and install that instead:
#   %pip install /lakehouse/default/Files/dp/pysqldp-0.1.0-py3-none-any.whl
# For a permanent install, add pysqldp to a Fabric Environment's public or
# custom libraries and attach the notebook to it.
%pip install pysqldp

import sys

print(sys.version)
assert sys.version_info >= (3, 11), (
    f"pysqldp needs Python >= 3.11, this runtime has {sys.version.split()[0]}. "
    "Attach this notebook to Fabric Runtime 1.3 (Spark 3.5) or later."
)

In [ ]:
# Cell 2 of 5 -- demo tables in the attached Lakehouse.
#
# Synthetic rows: 3 in north, 2 in south, none in west. 'west' exists in the
# domain table but has no events, so bin filling has to invent that bin -- an
# absent bin would itself leak that nobody is in west.
#
# Created only if missing, so re-running the notebook does not change the
# data under the answers.
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

EVENTS_SCHEMA = StructType(
    [
        StructField("region", StringType(), nullable=False),
        StructField("amount", DoubleType(), nullable=False),
    ]
)
DOMAIN_SCHEMA = StructType([StructField("region", StringType(), nullable=False)])

ROWS = [
    ("north", 1.0),
    ("north", 2.0),
    ("north", 3.0),
    ("south", 4.0),
    ("south", 5.0),
]
DOMAIN = [("north",), ("south",), ("west",)]

if not spark.catalog.tableExists("events"):
    spark.createDataFrame(ROWS, EVENTS_SCHEMA).write.format("delta").saveAsTable(
        "events"
    )
if not spark.catalog.tableExists("region_domain"):
    spark.createDataFrame(DOMAIN, DOMAIN_SCHEMA).write.format("delta").saveAsTable(
        "region_domain"
    )

spark.sql("SELECT region, COUNT(*) AS n FROM events GROUP BY 1 ORDER BY 1").show()

In [ ]:
# Cell 3 of 5 -- load the DP schema.
#
# The schema is what makes a query answerable: which columns may appear in a
# GROUP BY (canRelease), where each one's full domain lives (domainSet), and
# the clip bounds a SUM needs (clipLower/clipUpper). Upload
# tests/fixtures/schema_fabric.yaml to the Lakehouse Files area as
# Files/dp/schema.yaml; the notebook sees it at the POSIX path below.
#
# Two keys carry the Fabric specifics:
#   dialect: "spark"   -- emit Spark SQL. Required: Spark's RAND() is drawn per
#                         row, which is what the Laplace mechanism needs.
#   namespace: ""      -- unqualified names, resolved against this Lakehouse.
from pysqldp.schema import Schema

SCHEMA_PATH = "/lakehouse/default/Files/dp/schema.yaml"

db = Schema.load(SCHEMA_PATH).database("fabric")
print("database:", db.database)
print("sqlglot dialect:", db.sqlglot_dialect)
print("tables:", db.table_names)
assert db.sqlglot_dialect == "spark", 'the schema must set dialect: "spark"'

In [ ]:
# Cell 4 of 5 -- rewrite, then run.
#
# rewrite_fixed_pipeline returns an *intrinsically private* query: clipping,
# bin filling and the Laplace noise all live inside the SQL, so Spark runs it
# unchanged and the engine needs to know nothing about privacy.
#
# epsilon_spent is the budget this one execution costs. Deducting it from the
# analyst's allowance is the caller's job: pysqldp is stateless, and running
# the query a second time spends the budget a second time.
#
# epsilon = 1.0 is chosen so the demo is readable, not because it is a
# defensible production value. Real deployments sit nearer 0.1, where noise
# dominates small counts entirely.
from pysqldp import rewrite_fixed_pipeline

QUERY = "SELECT region, SUM(amount) AS total FROM events GROUP BY 1"
EPSILON = 1.0

sql, epsilon_spent = rewrite_fixed_pipeline(db, QUERY, EPSILON)
print("private SQL:\n", sql, "\n")
print("epsilon spent:", epsilon_spent)

print("\nprivate answer:")
spark.sql(sql).orderBy("region").show()

# Shown only because this is a demo on synthetic data. On real data the true
# answer is the thing being protected and must not be displayed beside the
# private one -- their difference is the noise, and revealing it undoes the
# mechanism.
print("true answer (synthetic data only):")
spark.sql(QUERY).orderBy("region").show()

In [ ]:
# Cell 5 of 5 -- a refusal.
#
# Not every query can be made private. A query returning raw rows has no
# aggregate to hide anybody behind, so there is no sound amount of noise to
# add: pysqldp refuses it rather than returning a weakly noised answer that
# looks private.
from pysqldp import UnsupportedQueryException

try:
    rewrite_fixed_pipeline(db, "SELECT region, amount FROM events", EPSILON)
except UnsupportedQueryException as exc:
    print("refused, as it should be:", exc)
else:
    raise AssertionError("a non-aggregate query must not be rewritten")